# 02 — Data Analysis & Cleaning (Phase 2, Pre-processing)

Covers:
1. Source files, schema, units, missing values, duplicates, quality checks.
2. Exclusions, encodings, transformations (documented).
3. Leak-safe transformations (fit on training split only).

All outputs are written to `outputs/tables/` and `data/processed/`.

In [1]:
import sys, json, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from Python.data_loader import LendingClubLoader
from Python.preprocessing import (
    LendingClubPreprocessor,
    derive_classification_target,
    leakage_columns,
    UNKNOWN_STATUSES,
)
from Python.splits import ChronologicalSplitter

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
OUT_TABLES = PROJECT_ROOT / 'outputs' / 'tables'
OUT_TABLES.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
PROJECT_ROOT

PosixPath('/Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados')

## 1. Load the raw files

We load the **accepted** and **rejected** files fully. This may take a while for
the accepted file (~2.2M rows, 151 cols). If you are memory-constrained, set
`usecols=` on the loader to a curated subset — but keep the same column list
for both the schema report and the cleaned parquet so results are reproducible.

In [2]:
ACCEPTED = DATA_RAW / 'accepted_2007_to_2018Q4.csv'
REJECTED = DATA_RAW / 'rejected_2007_to_2018Q4.csv'

loader = LendingClubLoader(ACCEPTED, REJECTED)
acc = loader.load_accepted()
rej = loader.load_rejected()
print('accepted:', acc.shape)
print('rejected:', rej.shape)

INFO | Loading accepted file: /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/data/raw/accepted_2007_to_2018Q4.csv
INFO | Accepted shape: (2260701, 151)
INFO | Loading rejected file: /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/data/raw/rejected_2007_to_2018Q4.csv
INFO | Rejected shape: (27648741, 9)


accepted: (2260701, 151)
rejected: (27648741, 9)


## 2. Schema reports

One report per file: dtypes, missing counts and %, n_unique, duplicate rows.

In [3]:
acc_report = loader.schema_report(acc, 'accepted')
rej_report = loader.schema_report(rej, 'rejected')

acc_report.to_frame().to_csv(OUT_TABLES / 'schema_accepted.csv')
rej_report.to_frame().to_csv(OUT_TABLES / 'schema_rejected.csv')

acc_report.to_frame().head(20)

,dtype,missing,missing_pct,n_unique
member_id,float64,2260701,100.0000,0
orig_projected_additional_accrued_interest,float64,2252050,99.6173,7487
hardship_end_date,str,2249784,99.5171,28
hardship_start_date,str,2249784,99.5171,27
hardship_type,str,2249784,99.5171,1
hardship_reason,str,2249784,99.5171,9
hardship_status,str,2249784,99.5171,3
deferral_term,float64,2249784,99.5171,1
hardship_last_payment_amount,float64,2249784,99.5171,9045
hardship_payoff_balance_amount,float64,2249784,99.5171,10893


In [4]:
rej_report.to_frame()

,dtype,missing,missing_pct,n_unique
Risk_Score,float64,18584384,67.2160,691
Debt-To-Income Ratio,float64,1203063,4.3512,126144
Employment Length,str,951355,3.4409,11
Loan Title,str,1305,0.0047,73927
Policy Code,float64,918,0.0033,2
Zip Code,str,293,0.0011,1001
State,str,22,0.0001,51
Amount Requested,float64,0,0.0000,3640
Application Date,datetime64[us],0,0.0000,4238


## 3. Duplicates and identifier checks

- Drop exact duplicates.
- Check `id` uniqueness in accepted.
- Document any colliding `(loan_amnt, issue_d, annual_inc)` triples.

In [5]:
dups_acc = acc.duplicated().sum()
dups_rej = rej.duplicated().sum()
print('exact duplicate rows (accepted):', dups_acc)
print('exact duplicate rows (rejected):', dups_rej)

acc = acc.drop_duplicates().reset_index(drop=True)
rej = rej.drop_duplicates().reset_index(drop=True)

id_dups = acc['id'].duplicated().sum() if 'id' in acc.columns else None
print('duplicate ids in accepted:', id_dups)

triple_dups = acc.duplicated(subset=['loan_amnt','issue_d','annual_inc']).sum()
print('possible repeated loans (loan_amnt, issue_d, annual_inc):', triple_dups)

exact duplicate rows (accepted): 0
exact duplicate rows (rejected): 157960
duplicate ids in accepted: 0
possible repeated loans (loan_amnt, issue_d, annual_inc): 1278624


## 4. Rejected file — quality checks

Per the spec:
- `Risk_Score == 0.0` is missing, not a genuine low score (already handled by the loader).
- `Debt-To-Income Ratio` is a text percentage; the loader parses it.
- Negative DTI sentinels become missing during parsing. As a conservative fixed cleaning policy, rejected DTI >100% is set missing, not used to exclude applications; genuinely high ratios may therefore also be lost.
- `Zip Code` is coerced to a zero-padded 5-char string.

In [6]:
print('Risk_Score missing (after 0 -> NaN):', rej['Risk_Score'].isna().sum())
print('DTI describe:')
print(rej['Debt-To-Income Ratio'].describe())

artifact_dti = rej['Debt-To-Income Ratio'] > 100  # 100% is already extreme
print('DTI > 100%:', int(artifact_dti.sum()))
rej.loc[artifact_dti, 'Debt-To-Income Ratio'] = np.nan

print('Zip sample:', rej['Zip Code'].head().tolist())

Risk_Score missing (after 0 -> NaN): 18445892
DTI describe:
count    2.632504e+07
mean     1.498097e+02
std      1.079805e+04
min      0.000000e+00
25%      9.710000e+00
50%      2.105000e+01
75%      3.752000e+01
max      5.000003e+07
Name: Debt-To-Income Ratio, dtype: float64
DTI > 100%: 805217
Zip sample: ['481xx', '010xx', '212xx', '017xx', '209xx']


## 5. Accepted file — missing values and quality checks

We compute a per-column missingness table and inspect the top offenders.
We do **not** impute yet — imputation is a *learned* transform and will be
fit on the training split only (see §7).

In [7]:
missing = pd.DataFrame({
    'missing': acc.isna().sum(),
    'missing_pct': (acc.isna().mean()*100).round(3),
}).sort_values('missing_pct', ascending=False)
missing.to_csv(OUT_TABLES / 'missingness_accepted.csv')
missing.head(25)

,missing,missing_pct
member_id,2260701,100.000
orig_projected_additional_accrued_interest,2252050,99.617
hardship_end_date,2249784,99.517
hardship_start_date,2249784,99.517
hardship_type,2249784,99.517
hardship_reason,2249784,99.517
hardship_status,2249784,99.517
deferral_term,2249784,99.517
hardship_last_payment_amount,2249784,99.517
hardship_payoff_balance_amount,2249784,99.517


### 5.1 Joint/secondary-applicant fields

Per the spec, these are **not** automatically leakage, but they are only
applicable to joint applications. We must distinguish *non-applicability*
from *missing information*. We do this by checking `application_type`.

In [8]:
if 'application_type' in acc.columns:
    print(acc['application_type'].value_counts(dropna=False))

joint_cols = [c for c in acc.columns if c.endswith('_joint') or c.startswith('sec_app_')]
joint_cols[:20]

application_type
Individual    2139958
Joint App      120710
NaN                33
Name: count, dtype: int64


['annual_inc_joint',
 'dti_joint',
 'verification_status_joint',
 'revol_bal_joint',
 'sec_app_fico_range_low',
 'sec_app_fico_range_high',
 'sec_app_earliest_cr_line',
 'sec_app_inq_last_6mths',
 'sec_app_mort_acc',
 'sec_app_open_acc',
 'sec_app_revol_util',
 'sec_app_open_act_il',
 'sec_app_num_rev_accts',
 'sec_app_chargeoff_within_12_mths',
 'sec_app_collections_12_mths_ex_med',
 'sec_app_mths_since_last_major_derog']

In [9]:
# Non-applicability = NaN when application_type == 'Individual'.
# Do NOT replace with 0. We keep the NaNs and let the pipeline impute
# inside the training fold only.
if 'application_type' in acc.columns and joint_cols:
    joint_missingness = acc.groupby('application_type')[joint_cols].apply(
        lambda g: g.isna().mean()
    ).T
    joint_missingness.to_csv(OUT_TABLES / 'joint_missingness_by_application_type.csv')
    joint_missingness.head(20)

## 6. Targets, cohorts, and exclusions

### 6.1 Classification target
Map `loan_status` to {0: Good, 1: At risk, 2: Default}. Drop `Current`.

### 6.2 Regression cohort
Keep only rows with a non-null `int_rate`. No further filtering needed here.

### 6.3 Leakage columns
Record what is being dropped, per task, in `outputs/tables/exclusions_log.csv`.

In [10]:
acc['target_class'] = derive_classification_target(acc['loan_status'])
print(acc['target_class'].value_counts(dropna=False))

print('\nCurrent (unknown) rows:', acc['loan_status'].isin(UNKNOWN_STATUSES).sum())

target_class
0       1076751
<NA>     881099
2        268599
1         34252
Name: count, dtype: Int64

Current (unknown) rows: 878317


In [11]:
# Regression cohort: must have int_rate
acc_reg = acc[acc['int_rate'].notna()].copy()

# Classification cohort: must have a known target
acc_clf = acc[acc['target_class'].notna()].copy()

print('regression cohort:', acc_reg.shape)
print('classification cohort:', acc_clf.shape)

regression cohort: (2260668, 152)
classification cohort: (1379602, 152)


In [12]:
# Leakage columns present in the dataset
leak = leakage_columns(acc)
exclusions_log = pd.DataFrame({'dropped_column': leak, 'reason': 'post-origination leakage'})
exclusions_log.to_csv(OUT_TABLES / 'exclusions_log.csv', index=False)
len(leak), leak[:15]

(33,
 ['collection_recovery_fee',
  'debt_settlement_flag',
  'debt_settlement_flag_date',
  'hardship_amount',
  'hardship_dpd',
  'hardship_end_date',
  'hardship_flag',
  'hardship_last_payment_amount',
  'hardship_length',
  'hardship_loan_status',
  'hardship_payoff_balance_amount',
  'hardship_reason',
  'hardship_start_date',
  'hardship_status',
  'hardship_type'])

## 7. Chronological split + leak-safe preprocessing

We apply the split designed in Notebook 01 using `issue_d`, then demonstrate
fitting a `ColumnTransformer` on the **training split only**.

The pipeline is not run over the test split for fitting — only for transforming —
which guarantees fold-safe preprocessing.

In [13]:
with open(PROJECT_ROOT / 'outputs' / 'split_definition.json') as f:
    split_cfg = json.load(f)
split_cfg

{'date_col': 'issue_d', 'train_end': '2014-12', 'val_end': '2016-12'}

In [14]:
splitter = ChronologicalSplitter(**split_cfg)

# Regression split
splits_reg = splitter.split(acc_reg)
print('regression split sizes:',
      len(splits_reg.train), len(splits_reg.val), len(splits_reg.test))

# Classification split
splits_clf = splitter.split(acc_clf)
print('classification split sizes:',
      len(splits_clf.train), len(splits_clf.val), len(splits_clf.test))

regression split sizes: 466345 855502 938821
classification split sizes: 451671 678142 249789


### 7.1 Define the numeric and categorical column lists

Only columns whose values are known at the prediction time are included.
Predictors are explicitly allowlisted below; `drop_leakage()` is also called before selection. Persisted source cohorts retain excluded fields for audit/target derivation, so they must never be passed wholesale to a model.

In [15]:
numeric_cols = [
    'loan_amnt', 'term', 'annual_inc', 'dti', 'fico_range_low', 'fico_range_high',
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'total_acc', 'pub_rec',
    'pub_rec_bankruptcies', 'mort_acc', 'revol_bal', 'revol_util',
    'tot_cur_bal', 'tot_hi_cred_lim', 'bc_util', 'bc_open_to_buy',
    'mo_sin_old_rev_tl_op', 'num_actv_rev_tl', 'pct_tl_nvr_dlq',
]
categorical_cols = [
    'home_ownership', 'verification_status', 'purpose', 'addr_state',
    'application_type',
]

# Keep only columns actually present in the data
numeric_cols = [c for c in numeric_cols if c in acc_clf.columns]
categorical_cols = [c for c in categorical_cols if c in acc_clf.columns]
numeric_cols, categorical_cols

(['loan_amnt',
  'term',
  'annual_inc',
  'dti',
  'fico_range_low',
  'fico_range_high',
  'delinq_2yrs',
  'inq_last_6mths',
  'open_acc',
  'total_acc',
  'pub_rec',
  'pub_rec_bankruptcies',
  'mort_acc',
  'revol_bal',
  'revol_util',
  'tot_cur_bal',
  'tot_hi_cred_lim',
  'bc_util',
  'bc_open_to_buy',
  'mo_sin_old_rev_tl_op',
  'num_actv_rev_tl',
  'pct_tl_nvr_dlq'],
 ['home_ownership',
  'verification_status',
  'purpose',
  'addr_state',
  'application_type'])

### 7.2 Fit on training, transform validation and test

This is the only place where learned transforms see data, and only the training
portion. The test portion is only `.transform()`-ed.

In [16]:
# Build a classification preprocessor as the demo (same pattern for regression)
pre = LendingClubPreprocessor(
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    task='classification',
)

X_train = pre.drop_leakage(splits_clf.train)[numeric_cols + categorical_cols]
X_val   = pre.drop_leakage(splits_clf.val)[numeric_cols + categorical_cols]
X_test  = pre.drop_leakage(splits_clf.test)[numeric_cols + categorical_cols]

X_train_t = pre.fit_transform(X_train)  # learned on train only
X_val_t   = pre.transform(X_val)        # transform only
X_test_t  = pre.transform(X_test)       # transform only

print('shapes:', X_train_t.shape, X_val_t.shape, X_test_t.shape)

INFO | Dropping 39 banned columns for task=classification
INFO | Dropping 39 banned columns for task=classification
INFO | Dropping 39 banned columns for task=classification


shapes: (451671, 96) (678142, 96) (249789, 96)


## 8. Persist cleaned data and split IDs

We save the cleaned accepted and rejected data as Parquet, plus a small
`split_ids.json` describing the cutoffs and per-split row counts, so Phase 3+
can reload exactly the same split.

In [17]:
acc_reg.to_parquet(DATA_PROCESSED / 'accepted_regression.parquet', index=False)
acc_clf.to_parquet(DATA_PROCESSED / 'accepted_classification.parquet', index=False)
rej.to_parquet(DATA_PROCESSED / 'rejected_clean.parquet', index=False)

split_ids = {
    'regression': {
        'cutoffs': splits_reg.cutoffs,
        'n_train': len(splits_reg.train),
        'n_val':   len(splits_reg.val),
        'n_test':  len(splits_reg.test),
    },
    'classification': {
        'cutoffs': splits_clf.cutoffs,
        'n_train': len(splits_clf.train),
        'n_val':   len(splits_clf.val),
        'n_test':  len(splits_clf.test),
    },
}
with open(PROJECT_ROOT / 'outputs' / 'split_ids.json', 'w') as f:
    json.dump(split_ids, f, indent=2)
split_ids

{'regression': {'cutoffs': {'train_end': '2014-12-31T00:00:00',
   'val_end': '2016-12-31T00:00:00',
   'test_start': '2017-01-01T00:00:00'},
  'n_train': 466345,
  'n_val': 855502,
  'n_test': 938821},
 'classification': {'cutoffs': {'train_end': '2014-12-31T00:00:00',
   'val_end': '2016-12-31T00:00:00',
   'test_start': '2017-01-01T00:00:00'},
  'n_train': 451671,
  'n_val': 678142,
  'n_test': 249789}}

## 9. Deterministic sampling for future EDA

The spec explicitly forbids using the first rows of a file as a representative
sample. We provide a reproducible sampler here to be reused in EDA and Phase 3.

In [18]:
from Python.eda_utils import LendingClubEDA
eda = LendingClubEDA()

sample_check = eda.reproducible_sample(splits_reg.train, n=1000, seed=42)
sample_check.head(3)

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term,target_class
1119101,37690417,NaN,15525.0,15525.0,15525.0,36.0,8.67,491.31,B,B1,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0
1119339,37329171,NaN,35000.0,35000.0,35000.0,60.0,17.14,872.48,D,D4,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,2
1119531,37780152,NaN,10800.0,10800.0,10800.0,36.0,16.49,382.32,D,D3,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0


## 10. Summary of decisions (for the mid-journey report)

- Deterministic parsing (percent signs, dates, zip) is done in the loader.
- Exact duplicate rows removed from both files; duplicate `id`s flagged.
- `Risk_Score == 0` treated as missing; DTI > 100% treated as an artifact.
- Joint-applicant fields kept as-is with NaNs where non-applicable.
- Leakage columns enumerated in `exclusions_log.csv` and dropped by the preprocessor.
- Split: train ≤ 2014-12, val 2015-01..2016-12, test 2017-01+ (per `split_definition.json`).
- Learned transforms fit on training only via `LendingClubPreprocessor`.
- Cleaned data + split IDs saved under `data/processed/` and `outputs/`.
- Negative accepted DTI and the fixed 999 sentinel become NaN before analysis; negative rejected DTI is likewise missing. Legitimate positive extremes remain visible and are not automatically winsorized.
- Verification is an origination-time assumption for this classifier demonstration; it is excluded from the separate pre-pricing regression plan.
- Temporal evaluation uses retrospective labels; neither a common outcome cutoff nor historical label availability has been established.